# Week 2 Day 2 Drill: Tone Rewriter

Build a Gradio app with Gemini:

1. **Inputs**: a textbox (the text to rewrite) and a dropdown with `"formal"`, `"funny"`, `"pirate"` (default: `"formal"`)
2. **Output**: a Markdown box
3. **Function** `rewrite(text, tone)`:
   - system prompt tells the model to rewrite the user's text in the chosen tone (use an f-string to put `tone` into it)
   - stream the answer from Gemini
   - `yield` the accumulated text
4. Launch it with `gr.Interface`

**Check yourself:** does your function take arguments in the same order as `inputs=[...]`? Are you yielding the full text so far?


In [ ]:
import os
from sys import flags

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI


In [ ]:
load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")
model = "gemini-3.1-flash-lite"

gemini = OpenAI(
    base_url=gemini_base_url,
    api_key=gemini_api_key
)



In [ ]:
def callgemini(prompt, tone):
    system_message = f"Rewrite the user's text in a {tone} tone. Reply with only the rewritten text."
    messages = [
        {"role" : "system", "content" : system_message},
        {"role" : "user", "content" : prompt}
    ]
    response = gemini.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result



In [ ]:
message_inputs = [
 gr.Textbox(),
 gr.Dropdown(
     choices=["formal", "funny", "pirate"],
     value="formal"
 )
]
message_output = gr.Markdown(label="Gemini")

view = gr.Interface(
    fn=callgemini,
    inputs=message_inputs,
    outputs=message_output,
    )
view.launch()


# Week 2 Day 3 Drill: Cooking Assistant Chatbot

Build a chatbot with `gr.ChatInterface` and Gemini:

1. **System prompt** (triple-quoted): a friendly cooking assistant. Include one fact (e.g. "today's special ingredient is mushrooms, suggest it when relevant") and one example reply
2. **Function** `chat(message, history)`:
   - clean the history (keep only `role` and `content`); a normal for loop is fine
   - build `[system] + history + [new message]`
   - stream the reply from Gemini and `yield` the accumulated text
3. **Dynamic rule**: if the message mentions "vegan", add an instruction to a *copy* of the system prompt telling the model to only suggest vegan recipes
4. Launch with `gr.ChatInterface(fn=chat, type="messages")`

**Test it:** ask for a recipe, then ask a follow-up like "can you make it spicier?". If it knows what "it" is, your history works.

In [18]:
new_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [26]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    system_message = new_message
    if "belt" in message.lower():
        system_message+="we dont have belts"
        message = [[{"role" : "system", "content" : system_message} ]+ history + [{"role" : "user", "content" : message}]]

    stream = gemini.chat.completions.create(
        model=model,
        messages=message,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [27]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\Asweddrtt\PycharmProjects\ai_engineering_exercises\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Asweddrtt\PycharmProjects\ai_engineering_exercises\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Asweddrtt\PycharmProjects\ai_engineering_exercises\.venv\Lib\site-packages\gradio\blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Asweddrtt\PycharmProjects\ai_engineering_exercises\.venv\Lib\site-packages\gradio\blocks.py", line 1710, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Use